# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python)

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹1 Comprensión y Preparación de los Datos

---

### 1.1 Carga de Datos

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**

- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [ ]:
# importar librerías
import pandas as pd
import numpy as nm
from scipy import stats
from scipy.stats import ttest_ind
from scipy.stats import chi2_contingency
from statsmodels.stats.proportion import proportions_ztest

In [ ]:
# cargar archivos
orders = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv')
catalog = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv')
marketing = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv')

In [ ]:
orders.info()
orders.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), object(8)
memory usage: 2.3+ MB


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28


In [ ]:
catalog.info()
catalog.head(7)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object(3)
memory usage: 352.0+ bytes


,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"
5,Sneakers-Urban-42,Moda,17.21,Greene-Smith
6,Jacket-Winter-M,Moda,189.31,Mcmillan-Rhodes


In [ ]:
marketing.info()
marketing.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   object 
 1   pais        1620 non-null   object 
 2   id_campaña  1620 non-null   object 
 3   canal       1519 non-null   object 
 4   gasto       1620 non-null   float64
dtypes: float64(1), object(4)
memory usage: 63.4+ KB


,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,organic_Mexico,organic,2446.25
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.34
2,2025-01-01,Mexico,social_Mexico,social,2045.01
3,2025-01-01,Colombia,organic_Colombia,organic,2597.21
4,2025-01-01,Colombia,paid_search_Colombia,paid_search,1771.40


### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas

---

In [ ]:
print("=== ORDERS ===")
display(orders.describe())
orders['fecha_hora_pedido'] = pd.to_datetime(orders['fecha_hora_pedido'], errors="coerce")
for col in ['cantidad',
            'precio_unitario',
            'monto_descuento',
            'monto_total']:
    print(f'\n{col}')
    print('Negativos:', (orders[col] < 0).sum())
    print('Ceros:', (orders[col] == 0).sum())

print("=== MARKETING ===")
display(marketing.describe())
marketing['fecha'] = pd.to_datetime(marketing['fecha'], errors="coerce")
print('Negativos gasto:',
      (marketing['gasto'] < 0).sum())
print('Ceros gasto:',
      (marketing['gasto'] == 0).sum())

print("=== CATALOG ===")
display(catalog.describe())
print('Negativos costo_unitario:',
     (catalog['costo_unitario'] < 0).sum())
print('Ceros costo_unitario:',
     (catalog['costo_unitario'] == 0).sum())


=== ORDERS ===


,cantidad,precio_unitario,monto_descuento,monto_total
count,25050.000000,25050.000000,25050.000000,2.510000e+04
mean,7.092735,259.305549,4.500798,2.072680e+03
std,296.277003,138.726461,5.223010,9.894995e+04
min,-2.000000,20.030000,0.000000,-4.926500e+02
25%,1.000000,138.377500,0.000000,1.805075e+02
50%,2.000000,258.715000,0.000000,3.417500e+02
75%,2.000000,380.332500,10.000000,5.185800e+02
max,20000.000000,499.960000,15.000000,8.840200e+06



cantidad
Negativos: 4
Ceros: 0

precio_unitario
Negativos: 0
Ceros: 0

monto_descuento
Negativos: 0
Ceros: 12551

monto_total
Negativos: 4
Ceros: 0
=== MARKETING ===


,gasto
count,1620.00000
mean,1772.74292
std,734.43294
min,501.11000
25%,1128.03000
50%,1782.42500
75%,2420.68500
max,2999.36000


Negativos gasto: 0
Ceros gasto: 0
=== CATALOG ===


,costo_unitario
count,7.000000
mean,102.252857
std,111.011563
min,10.120000
25%,16.905000
50%,25.210000
75%,182.975000
max,280.680000


Negativos costo_unitario: 0
Ceros costo_unitario: 0


In [ ]:
orders['monto_calculado'] = (
    orders['cantidad'] *
    orders['precio_unitario']
) - orders['monto_descuento']

orders['monto_calculado'] = orders['monto_calculado'].round(2)

inconsistencias = orders[
    orders['monto_total'].round(2) !=
    orders['monto_calculado']
]

print(f'Registros inconsistentes: {len(inconsistencias)}')
inconsistencias[
    ['id_pedido',
     'cantidad',
     'precio_unitario',
     'monto_descuento',
     'monto_total',
     'monto_calculado']
].head()
orders[orders['monto_total'] < 0]

Registros inconsistentes: 6309


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total,monto_calculado
266,order_266,user_7011,2025-03-13,NaN,desktop,paid_search,Phone-Pro-128GB,Electronica,-2.0,101.31,10.0,-192.62,-212.62
267,order_267,user_1087,2025-05-07,NaN,desktop,social,Phone-Pro-128GB,Electronica,-1.0,43.50,5.0,-38.50,-48.50
268,order_268,user_84,2025-02-19,NaN,desktop,organic,Phone-Pro-128GB,Electronica,-1.0,497.65,5.0,-492.65,-502.65
269,order_269,user_3323,2025-05-25,NaN,desktop,paid_search,Phone-Pro-128GB,Electronica,-1.0,423.53,0.0,-423.53,-423.53


In [ ]:
orders.drop(columns=['monto_calculado'], inplace=True)
orders.columns

Index(['id_pedido', 'id_usuario', 'fecha_hora_pedido', 'pais', 'dispositivo',
       'fuente_referencia', 'nombre_producto', 'categoria_producto',
       'cantidad', 'precio_unitario', 'monto_descuento', 'monto_total'],
      dtype='object')

In [ ]:
orders.duplicated().sum()

100

In [ ]:
orders[orders.duplicated(keep=False)] \
.sort_values('id_pedido') \
.head(20)

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
25023,order_10082,user_690,2025-02-17,Argentina,desktop,social,Sneakers-Urban-42,Moda,2.0,221.34,0.0,442.67
10082,order_10082,user_690,2025-02-17,Argentina,desktop,social,Sneakers-Urban-42,Moda,2.0,221.34,0.0,442.67
25037,order_10709,user_6783,2025-02-16,Colombia,mobile,organic,Jacket-Winter-M,Moda,1.0,170.10,10.0,160.10
10709,order_10709,user_6783,2025-02-16,Colombia,mobile,organic,Jacket-Winter-M,Moda,1.0,170.10,10.0,160.10
25065,order_10829,user_7697,2025-01-23,Argentina,mobile,social,Phone-Pro-128GB,Electronica,2.0,115.54,0.0,231.08
10829,order_10829,user_7697,2025-01-23,Argentina,mobile,social,Phone-Pro-128GB,Electronica,2.0,115.54,0.0,231.08
10856,order_10856,user_1899,2025-03-18,Colombia,desktop,social,Laptop-Gaming-16GB,Electronica,1.0,72.53,0.0,72.53
25044,order_10856,user_1899,2025-03-18,Colombia,desktop,social,Laptop-Gaming-16GB,Electronica,1.0,72.53,0.0,72.53
11449,order_11449,user_1480,2025-03-21,Mexico,desktop,paid_search,Vacuum-Pro-Black,Hogar,1.0,60.55,5.0,55.55
25083,order_11449,user_1480,2025-03-21,Mexico,desktop,paid_search,Vacuum-Pro-Black,Hogar,1.0,60.55,5.0,55.55


In [ ]:
filas_antes = len(orders)

orders = orders.drop_duplicates()

filas_despues = len(orders)

print("Filas eliminadas:", filas_antes - filas_despues)
print("Duplicados restantes:", orders.duplicated().sum())

orders.info()

Filas eliminadas: 100
Duplicados restantes: 0
<class 'pandas.core.frame.DataFrame'>
Int64Index: 25000 entries, 0 to 24999
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_pedido           25000 non-null  object        
 1   id_usuario          25000 non-null  object        
 2   fecha_hora_pedido   25000 non-null  datetime64[ns]
 3   pais                24700 non-null  object        
 4   dispositivo         24980 non-null  object        
 5   fuente_referencia   24970 non-null  object        
 6   nombre_producto     24970 non-null  object        
 7   categoria_producto  24920 non-null  object        
 8   cantidad            24950 non-null  float64       
 9   precio_unitario     24950 non-null  float64       
 10  monto_descuento     24950 non-null  float64       
 11  monto_total         25000 non-null  float64       
dtypes: datetime64[ns](1), float64(4), object(7)
memory usage

In [ ]:
for col in orders.select_dtypes(include='object').columns:
    print(f'\n{col}')
    print('Valores únicos:', orders[col].nunique())
    print(orders[col].value_counts(dropna=False))

for col in marketing.select_dtypes(include='object').columns:
    print(f'\n{col}')
    print(marketing[col].value_counts(dropna=False))


id_pedido
Valores únicos: 25000
order_17919    1
order_20445    1
order_2963     1
order_18102    1
order_24244    1
              ..
order_9257     1
order_17659    1
order_24489    1
order_22211    1
order_5949     1
Name: id_pedido, Length: 25000, dtype: int64

id_usuario
Valores únicos: 7642
user_7769    11
user_6037    10
user_1468    10
user_3000    10
user_3267    10
             ..
user_4824     1
user_657      1
user_222      1
user_6011     1
user_2772     1
Name: id_usuario, Length: 7642, dtype: int64

pais
Valores únicos: 6
Colombia     7481
Mexico       7478
Argentina    7259
mexico        863
colombia      823
argentina     796
NaN           300
Name: pais, dtype: int64

dispositivo
Valores únicos: 2
desktop    12711
mobile     12269
NaN           20
Name: dispositivo, dtype: int64

fuente_referencia
Valores únicos: 3
social         8402
organic        8288
paid_search    8280
NaN              30
Name: fuente_referencia, dtype: int64

nombre_producto
Valores únicos: 7
Bl

In [ ]:
orders['pais'] = orders['pais'].str.title()
orders['pais'].value_counts(dropna=False)

Mexico       8341
Colombia     8304
Argentina    8055
NaN           300
Name: pais, dtype: int64

In [ ]:
orders['categoria_producto'] = (
    orders['categoria_producto']
    .fillna('Sin categoría')
)

orders['nombre_producto'] = (
    orders['nombre_producto']
    .fillna('Producto desconocido')
)
orders['fuente_referencia'] = (
    orders['fuente_referencia']
    .fillna('Referencia desconocido')
)

In [ ]:
print("Fecha mínima:", orders['fecha_hora_pedido'].min())
print("Fecha máxima:", orders['fecha_hora_pedido'].max())

Fecha mínima: 2025-01-01 00:00:00
Fecha máxima: 2025-06-30 00:00:00


In [ ]:
print(orders['fecha_hora_pedido'].dtype)

datetime64[ns]


---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [ ]:
# exportar datasets
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)

---

## 🔹 Análisis de Rentabilidad

### 2.1 Indicadores Clave del Negocio (KPIs)

Se evaluó el desempeño financiero del negocio mediante el análisis de ingresos, costos operativos, inversión en marketing y utilidad neta.

### Métricas Analizadas

- Revenue
- Costos Totales
- Inversión en Marketing
- Profit
- Margen de Rentabilidad 

---

Análisis del Comportamiento de Ventas

Se analizaron los patrones de compra para identificar oportunidades de crecimiento y optimización comercial.

### Indicadores Evaluados

- Ticket promedio por orden
- Productos por transacción
- Productos con mayor volumen de ventas
- Distribución de ventas por categoría
- Inversión de marketing por canal

In [ ]:
ingresos_totales = orders['monto_total'].sum()
print(f'Ingresos totales: ${ingresos_totales:,.2f}')

orders_profit = orders.merge(
    catalog[['nombre_producto', 'costo_unitario']],
    on='nombre_producto',
    how='left'
)
orders_profit['costo_total_pedido'] = (
    orders_profit['cantidad'] *
    orders_profit['costo_unitario']
)
costo_total = orders_profit['costo_total_pedido'].sum()
print(f'Costo total: ${costo_total:,.2f}')

inversion_marketing = marketing['gasto'].sum()
print(f'Inversión en marketing: ${inversion_marketing:,.2f}')

profit = (
    ingresos_totales
    - costo_total
    - inversion_marketing
)

print(f'Profit: ${profit:,.2f}')

Ingresos totales: $51,987,462.26
Costo total: $43,124,018.41
Inversión en marketing: $2,871,843.53
Profit: $5,991,600.32


In [ ]:
ticket_promedio = orders['monto_total'].mean()
print(f'Ticket promedio: ${ticket_promedio:,.2f}')

cantidad_promedio = orders['cantidad'].mean()
print(f'Cantidad promedio de productos por orden: {cantidad_promedio:.2f}')

productos_vendidos = (
    orders.groupby('nombre_producto')['cantidad']
    .sum()
    .sort_values(ascending=False)
)
print(productos_vendidos)

marketing_canal = (
    marketing.groupby('canal')['gasto']
    .sum()
    .sort_values(ascending=False)
)
print(marketing_canal)

Ticket promedio: $2,079.50
Cantidad promedio de productos por orden: 7.12
nombre_producto
Laptop-Gaming-16GB      144198.0
Vacuum-Pro-Black          6284.0
Blender-XL-Red            6279.0
Jacket-Winter-M           6256.0
Sneakers-Urban-42         6172.0
Tablet-Standard-64GB      4153.0
Phone-Pro-128GB           4135.0
Producto desconocido        45.0
Name: cantidad, dtype: float64
canal
social         918043.21
organic        913533.01
paid_search    863088.21
Name: gasto, dtype: float64


---

## 🔹 3 Anáslisis del Funnel de Conversion

**🎯 Objetivo:** Analizar el recorrido de los usuarios a través del proceso de compra para identificar los puntos de abandono y las oportunidades de optimización de la conversión.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

3.1 Construcción del Funnel

Se construyó un funnel de conversión utilizando los eventos registrados en la plataforma para visualizar el flujo de usuarios desde la primera interacción hasta la compra.

Actividades realizadas:

- Identificación de las etapas clave del recorrido del usuario.
- Cálculo de usuarios únicos por evento.
- Ordenamiento secuencial de eventos según el flujo de navegación.
- Construcción del funnel de conversión.

---

3.2 Evaluación de Conversiones

Se analizaron las tasas de conversión entre etapas para determinar dónde se producen las mayores pérdidas de usuarios.

Métricas evaluadas:

- Conversión entre etapas consecutivas.
- Tasa de conversión acumulada.
- Tasa de abandono por etapa.
- Conversión final hacia la compra.

---

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [ ]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [ ]:
# PARTE 1: Totales del funnel
# ======================

query_totals = '''
SELECT
    nombre_evento,
    COUNT(DISTINCT id_usuario) AS usuarios_unicos
FROM events
GROUP BY nombre_evento
ORDER BY usuarios_unicos DESC;
'''

totals = pd.read_sql(query_totals, con=engine)
totals

,nombre_evento,usuarios_unicos
0,first_visit,7796
1,add_to_cart,7634
2,select_item,7582
3,begin_checkout,7208
4,add_payment_info,6250
5,purchase,6240


In [ ]:
# PARTE 2: Conversiones
# ======================

query_conversion = '''
WITH funnel AS (
    SELECT
        nombre_evento,
        COUNT(DISTINCT id_usuario) AS usuarios_unicos,
        CASE
            WHEN nombre_evento = 'first_visit' THEN 1
            WHEN nombre_evento = 'select_item' THEN 2
            WHEN nombre_evento = 'add_to_cart' THEN 3
            WHEN nombre_evento = 'begin_checkout' THEN 4
            WHEN nombre_evento = 'add_payment_info' THEN 5
            WHEN nombre_evento = 'purchase' THEN 6
        END AS orden_etapa
    FROM events
    GROUP BY nombre_evento
)

SELECT
    nombre_evento,
    usuarios_unicos,
    ROUND(
        usuarios_unicos * 100.0 /
        LAG(usuarios_unicos) OVER (ORDER BY orden_etapa),
        2
    ) AS conversion_porcentaje
FROM funnel
ORDER BY orden_etapa;
'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion

,nombre_evento,usuarios_unicos,conversion_porcentaje
0,first_visit,7796,NaN
1,select_item,7582,97.26
2,add_to_cart,7634,100.69
3,begin_checkout,7208,94.42
4,add_payment_info,6250,86.71
5,purchase,6240,99.84


---

## 🔹 4. Análisis de Retención de Clientes

Objetivo: Evaluar la capacidad de la plataforma para retener usuarios después de su registro, identificando patrones de permanencia y abandono a lo largo del tiempo.

**4.1 Metodología de Cohortes**

Se realizó un análisis de cohortes agrupando a los usuarios según su mes de registro. Esto permite comparar el comportamiento de distintos grupos de usuarios y medir la evolución de la retención durante las semanas posteriores a su incorporación.
| Tabla         | Descripción                         |
| ------------- | ----------------------------------- |
| users         | Información de registro de usuarios |
| user_activity | Actividad generada por los usuarios |


---
4.2 Cálculo de Retención

Para cada cohorte se calcularon las siguientes métricas:

Usuarios registrados por cohorte.
Usuarios activos por semana.
Tasa de retención semanal.
Evolución de la retención a lo largo del tiempo.

La retención se calculó como:

Retención = (Usuarios activos / Usuarios iniciales de la cohorte) * 100 

In [ ]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)


,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [ ]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT *
FROM user_activity
LIMIT 5;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)
user_activity.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 4 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   id_usuario             5 non-null      object
 1   fecha_actividad        5 non-null      object
 2   dias_despues_registro  5 non-null      int64 
 3   activo                 5 non-null      int64 
dtypes: int64(2), object(2)
memory usage: 288.0+ bytes


In [ ]:
# Explorar tabla user_activity
# =========================
query_users= '''
SELECT
    DATE_TRUNC(
        'month',
        CAST(fecha_registro AS DATE)
    ) AS fecha_registro
FROM users;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)
user_activity.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 4 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   id_usuario             5 non-null      object
 1   fecha_actividad        5 non-null      object
 2   dias_despues_registro  5 non-null      int64 
 3   activo                 5 non-null      int64 
dtypes: int64(2), object(2)
memory usage: 288.0+ bytes


In [ ]:
# Retención por cohortes
# ======================

query_cohort_retention_final = '''
WITH cohortes AS (

    SELECT
        id_usuario,
        DATE_TRUNC(
            'month',
            CAST(fecha_registro AS DATE)
        ) AS cohorte_mes
    FROM users

),

retencion AS (

    SELECT
        c.cohorte_mes,
        c.id_usuario,

        MAX(
            CASE
                WHEN ua.dias_despues_registro = 7
                     AND ua.activo = 1
                THEN 1 ELSE 0
            END
        ) AS retenido_w1,

        MAX(
            CASE
                WHEN ua.dias_despues_registro = 14
                     AND ua.activo = 1
                THEN 1 ELSE 0
            END
        ) AS retenido_w2,

        MAX(
            CASE
                WHEN ua.dias_despues_registro = 21
                     AND ua.activo = 1
                THEN 1 ELSE 0
            END
        ) AS retenido_w3

    FROM cohortes c

    LEFT JOIN user_activity ua
        ON c.id_usuario = ua.id_usuario

    GROUP BY
        c.cohorte_mes,
        c.id_usuario

)

SELECT

    cohorte_mes,
    COUNT(*) AS usuarios_iniciales,

    SUM(retenido_w1) AS retenido_w1,
    SUM(retenido_w2) AS retenido_w2,
    SUM(retenido_w3) AS retenido_w3,

    ROUND(
        SUM(retenido_w1) * 100.0 / COUNT(*),
        2
    ) AS semana_1,

    ROUND(
        SUM(retenido_w2) * 100.0 / COUNT(*),
        2
    ) AS semana_2,

    ROUND(
        SUM(retenido_w3) * 100.0 / COUNT(*),
        2
    ) AS semana_3

FROM retencion
GROUP BY cohorte_mes
ORDER BY cohorte_mes;
'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final

,cohorte_mes,usuarios_iniciales,retenido_w1,retenido_w2,retenido_w3,semana_1,semana_2,semana_3
0,2025-01-01 00:00:00+00:00,1627,697,668,656,42.84,41.06,40.32
1,2025-02-01 00:00:00+00:00,1444,611,609,635,42.31,42.17,43.98
2,2025-03-01 00:00:00+00:00,1636,677,705,690,41.38,43.09,42.18
3,2025-04-01 00:00:00+00:00,1606,680,697,663,42.34,43.40,41.28
4,2025-05-01 00:00:00+00:00,1687,695,676,706,41.20,40.07,41.85


---

## 🔹 5. Validación Estadística del Rediseño del Checkout

🎯 **Objetivo:** Determinar si la modificación implementada en la interfaz de checkout genera una mejora estadísticamente significativa en la tasa de conversión de compra.

---

5.1 Contexto del Experimento

Con el objetivo de mejorar la experiencia de usuario, se implementó una nueva versión de la interfaz de checkout. Para evaluar su efectividad, se comparó el comportamiento de los usuarios expuestos a la versión actual y a la versión modificada.

Métrica Principal

La métrica evaluada fue la tasa de conversión de compra, definida como:

1 = Usuario completó una compra.
0 = Usuario no completó una compra.  

---
Hipótesis estadística
   - **H₀ (Hipótesis nula):** La nueva interfaz del checkout no modifica la tasa de conversión.
   - **H₁ (Hipótesis alternativa):** La nueva interfaz del checkout modifica la tasa de conversión.
   
**Test estadístico:** -0.8132
**Nivel de significancia alpha:** 0.4160 No se rechaza H0

In [ ]:
experiment = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv')
experiment.head()

,id_usuario,variante,convirtio,dispositivo,pais,duracion_sesion,timestamp
0,exp_user_0,tratamiento,0,mobile,Argentina,114.41,2025-03-28
1,exp_user_1,tratamiento,0,desktop,Mexico,170.03,2025-01-15
2,exp_user_2,control,1,mobile,Colombia,140.21,2025-03-18
3,exp_user_3,tratamiento,0,mobile,Colombia,151.45,2025-06-03
4,exp_user_4,tratamiento,0,desktop,Mexico,299.96,2025-01-12


In [ ]:
experiment.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id_usuario       10000 non-null  object 
 1   variante         10000 non-null  object 
 2   convirtio        10000 non-null  int64  
 3   dispositivo      10000 non-null  object 
 4   pais             10000 non-null  object 
 5   duracion_sesion  10000 non-null  float64
 6   timestamp        10000 non-null  object 
dtypes: float64(1), int64(1), object(5)
memory usage: 547.0+ KB


In [ ]:
experiment.groupby('dispositivo')['convirtio'].agg(
    usuarios='count',
    conversion='mean'
)

,usuarios,conversion
dispositivo,,
desktop,5042,0.182864
mobile,4958,0.136547


In [ ]:
control = experiment[experiment['variante'] == 'control']
tratamiento = experiment[experiment['variante'] == 'tratamiento']
conversiones = [control['convirtio'].sum(), tratamiento['convirtio'].sum()]
muestras = [len(control),len(tratamiento)]

In [ ]:
stat, p_value = proportions_ztest(
    conversiones,
    muestras
)

print('Estadístico Z:', stat)
print('p-value:', p_value)

Estadístico Z: -0.8132782986429474
p-value: 0.41605851639119995


In [ ]:
alpha = 0.05
if p_value < alpha:
    print("Se rechaza H0")
else:
    print("No se rechaza H0")

No se rechaza H0


---

## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

🎯 **Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión.

Se usarán los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

---

1️⃣ Preparación de los datos
1. Cargar los CSVs en Power BI o Tableau.
2. Revisar relaciones:
   - `orders.nombre_producto` → `catalog.nombre_producto`
   - `orders.fecha_pedido` → tabla de fechas (crear calendario para análisis temporal)
   - `orders.fecha_pedido` → `dim_fecha.date`
3. Crear columnas calculadas necesarias
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores (`Previous Year`, `Previous Month`).

---

2️⃣ Dashboard 1: Overview Ejecutivo
**KPIs principales a mostrar:**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones sugeridas:**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico de líneas: evolución mensual de revenue o profit
- Gráfico de líneas YTD
- Gráfico de barras: revenue y profit por producto o categoría

---

 3️⃣ Dashboard 2: Detalle / Drill-through  
**Objetivo:** Permitir explorar los datos desde el KPI general hasta cada orden o producto.

**Visualizaciones sugeridas:**
- Tabla detallada de órdenes con:
  - producto, cantidad, revenue, cost, profit
  - color condicional (profit negativo en rojo, positivo en verde)
- Gráfico de barras por producto con medida `cantidad vendida`
- Drill-through: seleccionar un producto y ver todos los pedidos relacionados
- Filtros por fecha, categoría de producto, etc

---